# Bootstrap threshold analysis for scaffold-based and pharmacophore-based metrics

This notebook is the canonical entry point for bootstrap confidence intervals, pairwise bootstrap differences, and effect-size threshold estimation.

- Use `METRIC_FAMILY = "scaffold"` for scaffold-based metrics.
- Use `METRIC_FAMILY = "ph4"` for pharmacophore-based metrics.
- Intermediate outputs are stored in `../../data/bootstrap_threshold_analysis/...`.
- Only the final threshold CSV is stored in the repository root:
  - `effect_size_thresholds_scaffolds.csv`
  - `effect_size_thresholds_ph4_rdkit.csv`


In [18]:
import os
import importlib
from pathlib import Path

import pandas as pd

from src import bootstrap_threshold_analysis as bta
importlib.reload(bta)


<module 'src.bootstrap_threshold_analysis' from '/home/filv/phd_projects/iga_2023/git_reccal/new/diseration_git/generative_models_for_de_novo_molecular_design/src/bootstrap_threshold_analysis.py'>

## Configuration

Two parallelism knobs are available:

- `JOB_WORKERS`: how many `(receptor × split × unit × generator)` jobs run in parallel
- `INNER_WORKERS`: inner parallelism inside each job
  - scaffold workflow: SMILES to scaffold conversion
  - pharmacophore workflow: output-to-recall fingerprint matching

Recommended pattern:

- first run when caches are still being built: `JOB_WORKERS = 1`, high `INNER_WORKERS`
- later runs with warm caches: higher `JOB_WORKERS`, lower `INNER_WORKERS`


In [19]:
# ---- choose workflow ----
METRIC_FAMILY = "ph4"   # "scaffold" or "ph4"
DATA_FOLDER = "../../"

# ---- datasets ----
RECEPTORS = ["Glucocorticoid_receptor", "Leukocyte_elastase"]
SPLITS = ["dis", "sim"]
GENERATORS = bta.default_generators_for_family(METRIC_FAMILY)

# ---- family-specific units ----
if METRIC_FAMILY == "scaffold":
    UNITS = ["csk", "murcko"]
else:
    UNITS = ["rdkit"]

# ---- bootstrap parameters ----
CLUSTERS = [0, 1, 2, 3, 4]
N_SUBSAMPLE = 250_000
N_BOOTSTRAP = 300
ALPHA = 0.05

# ---- parallelism ----
CPU = 80
JOB_WORKERS = 40
INNER_WORKERS = max(1, CPU - 1)

# ---- performance / UX ----
USE_CACHE = True
SHOW_PROGRESS = True
CHUNKSIZE = 2000 if METRIC_FAMILY == "scaffold" else 200

cfg = bta.BootstrapThresholdConfig(
    metric_family=METRIC_FAMILY,
    units=UNITS,
    clusters=CLUSTERS,
    n_subsample=N_SUBSAMPLE,
    n_bootstrap=N_BOOTSTRAP,
    alpha=ALPHA,
    data_folder=DATA_FOLDER,
    job_workers=JOB_WORKERS,
    inner_workers=INNER_WORKERS,
    chunksize=CHUNKSIZE,
    use_cache=USE_CACHE,
    show_progress=SHOW_PROGRESS,
    save_bootstrap_samples=True,
)

INTERMEDIATE_ROOT = cfg.intermediate_root()
FINAL_CSV_PATH = cfg.default_final_csv()

print("Intermediate outputs:", INTERMEDIATE_ROOT)
print("Final threshold CSV:", FINAL_CSV_PATH)
cfg


Intermediate outputs: /home/filv/phd_projects/iga_2023/git_reccal/new/data/bootstrap_threshold_analysis/ph4
Final threshold CSV: /home/filv/phd_projects/iga_2023/git_reccal/new/diseration_git/generative_models_for_de_novo_molecular_design/effect_size_thresholds_ph4_rdkit.csv


BootstrapThresholdConfig(metric_family='ph4', units=['rdkit'], clusters=[0, 1, 2, 3, 4], n_subsample=250000, n_bootstrap=300, alpha=0.05, data_folder='../../', job_workers=40, inner_workers=79, chunksize=200, use_cache=True, show_progress=True, save_bootstrap_samples=True)

## Run the full workflow

This step:

1. computes bootstrap CIs for all jobs,
2. saves bootstrap distributions per job in `../../data`,
3. computes pairwise bootstrap differences,
4. derives effect-size thresholds,
5. writes the final CSV into the repository root.


In [20]:
df_summary, df_tests, df_thresholds = bta.run_full_threshold_workflow(
    receptors=RECEPTORS,
    splits=SPLITS,
    generators=GENERATORS,
    cfg=cfg,
    final_csv_path=FINAL_CSV_PATH,
)

df_thresholds


ph4 bootstrap jobs: 100%|██████████| 44/44 [1:13:25<00:00, 100.12s/job] 


[DONE] Wrote summary to /home/filv/phd_projects/iga_2023/git_reccal/new/data/bootstrap_threshold_analysis/ph4/bootstrap_ci_ph4_metrics_summary.csv (73.60 min)
[DONE] Wrote pairwise tests to /home/filv/phd_projects/iga_2023/git_reccal/new/data/bootstrap_threshold_analysis/ph4/bootstrap_pairwise_tests_ph4.csv
[DONE] Wrote final threshold CSV to /home/filv/phd_projects/iga_2023/git_reccal/new/diseration_git/generative_models_for_de_novo_molecular_design/effect_size_thresholds_ph4_rdkit.csv


,Metric,PH4,Trivial Δ (≤25%),Small Δ (25–50%),Moderate Δ (50–75%),Large Δ (>75%)
0,RS,rdkit,Δ ≤ 0.131,0.131 < Δ ≤ 0.290,0.290 < Δ ≤ 0.498,Δ > 0.498
1,SED,rdkit,Δ ≤ 0.027,0.027 < Δ ≤ 0.097,0.097 < Δ ≤ 0.151,Δ > 0.151
2,ASER,rdkit,Δ ≤ 0.004,0.004 < Δ ≤ 0.011,0.011 < Δ ≤ 0.019,Δ > 0.019


## Inspect saved outputs

The summary and pairwise test tables stay in `../../data`, while the final threshold CSV stays in the repository root.


In [21]:
display(pd.read_csv('/home/filv/phd_projects/iga_2023/git_reccal/new/diseration_git/generative_models_for_de_novo_molecular_design/effect_size_thresholds_scaffolds.csv'))
display(pd.read_csv('/home/filv/phd_projects/iga_2023/git_reccal/new/diseration_git/generative_models_for_de_novo_molecular_design/effect_size_thresholds_ph4_rdkit.csv'))


,Metric,Scaffold,Trivial Δ (≤25%),Small Δ (25–50%),Moderate Δ (50–75%),Large Δ (>75%)
0,RS,csk,Δ ≤ 0.072,0.072 < Δ ≤ 0.189,0.189 < Δ ≤ 0.332,Δ > 0.332
1,RS,murcko,Δ ≤ 0.044,0.044 < Δ ≤ 0.136,0.136 < Δ ≤ 0.212,Δ > 0.212
2,SED,csk,Δ ≤ 0.045,0.045 < Δ ≤ 0.090,0.090 < Δ ≤ 0.159,Δ > 0.159
3,SED,murcko,Δ ≤ 0.076,0.076 < Δ ≤ 0.175,0.175 < Δ ≤ 0.299,Δ > 0.299
4,ASER,csk,Δ ≤ 0.004,0.004 < Δ ≤ 0.011,0.011 < Δ ≤ 0.018,Δ > 0.018
5,ASER,murcko,Δ ≤ 0.001,0.001 < Δ ≤ 0.004,0.004 < Δ ≤ 0.007,Δ > 0.007


,Metric,PH4,Trivial Δ (≤25%),Small Δ (25–50%),Moderate Δ (50–75%),Large Δ (>75%)
0,RS,rdkit,Δ ≤ 0.131,0.131 < Δ ≤ 0.290,0.290 < Δ ≤ 0.498,Δ > 0.498
1,SED,rdkit,Δ ≤ 0.027,0.027 < Δ ≤ 0.097,0.097 < Δ ≤ 0.151,Δ > 0.151
2,ASER,rdkit,Δ ≤ 0.004,0.004 < Δ ≤ 0.011,0.011 < Δ ≤ 0.019,Δ > 0.019
